In [1]:
import numpy as np

In [2]:
# ============================
# Conversion factors
# ============================

GeVincm     = 5.068e13 # cm^-1 / GeV
GeVinsec    = 1.519e24 # s^-1 / GeV
KinGeV      = 8.62e-14 # GeV / K
GaussinGeV2 = 1.95e-20 # GeV^2 / G

# ============================
# Constants
# ============================

rSun = 8.277 # kpc
sigmaT = 0.6652e-24 # cm^2
me = 5.11e-4 # GeV
alpha = 1/137
e = (4*np.pi*alpha)**.5  # elementary charge in natural units

In [3]:
# ============================
# Photon field densities
# ============================

def nBB(Eg, T): # GeV^-1 cm^-3
    return GeVincm**3 / np.pi**2 * Eg**2 / (np.exp(Eg/T)-1)

def nISRF(Eg, R, z, T, R0, z0, n0):
    nTOT = GeVincm**3 / np.pi**2 * T**4 * np.pi**4/15
    return n0/nTOT * nBB(Eg, T) * np.exp(-abs(z)/z0) * np.exp(-(R-rSun)/R0)

def nOpt(Eg, R, z):
    return nISRF(Eg, R, z, 5e3*KinGeV, 2.5, 1.5, 0.50e-9)

def nIRw(Eg, R, z):
    return nISRF(Eg, R, z, 400*KinGeV, 3.5, 2, 0.05e-9)

def nIRc(Eg, R, z):
    return nISRF(Eg, R, z, 40*KinGeV,  2, 1, 0.20e-9)

def nCMB(Eg):
    return nBB(Eg, 2.725*KinGeV)

In [4]:
# ============================
# ICS radiating power
# ============================

def PIC(Eg, Ee, nISRF, *args): # GeV s^-1 GeV^-1

    gamma = Ee / me
    epsilon = Eg / Ee

    def integrand_log(lq, Eg, nISRF, *args):
        q = 10**lq
        Eg0 = Eg / (4 * gamma**2 * (1 - epsilon) * q)

        kernel = (2*q*np.log(q) + q + 1 - 2*q**2 + (epsilon**2 * (1 - q)) / (2 * (1-epsilon))) / q

        return (Eg - Eg0) * nISRF(Eg0, *args) * kernel * (epsilon < 1.0)

    Nlq = 200
    lqmin = np.log10(1/(4*gamma**2))
    lqmax = np.log10(1)

    lqs = np.linspace(lqmin, lqmax, Nlq)
    dlq = (lqmax - lqmin) / Nlq

    prefactor = GeVinsec/GeVincm * 3*sigmaT / (4*gamma**2)

    PIC = 0.0
    for lq in lqs:
        jac = 10**lq * dlq * np.log(10)
        PIC += prefactor * integrand_log(lq, Eg, nISRF, *args) * jac

    return PIC

In [5]:
# ==============================
# Magnetic field configurations
# ==============================

def BUni(B0): # Gauss
    return B0 

def BGal(R, z, B0, R0, z0):
    return B0 * np.exp(-(R-rSun)/R0) * np.exp(-np.abs(z)/z0)

def BMF1(R, z):
    return BGal(R, z, 4.78e-6, 10., 2)

def BMF2(R, z):
    return BGal(R, z, 5.10e-6, 8.5, 1)

def BMF3(R, z):
    return BGal(R, z, 9.50e-6, 30., 4)

In [6]:
# ==============================
# Synchrotron radiating power
# ==============================

from scipy.special import kv

def PSyn(nu, Ee, B, *args): # GeV s^-1 Hz^-1

    gamma = Ee/me
    nuc = GeVinsec * 3/(2*np.pi) * gamma**2 * e * B(*args)*GaussinGeV2 / me
    y = nu/nuc

    prefactor = 2 * 3**.5 * e**3 * B(*args)*GaussinGeV2 / me / (4*np.pi)
    kernel = y**2 * (kv(4/3, y)*kv(1/3, y) - 3/5*y * (kv(4/3, y)**2 - kv(1/3, y)**2))

    return prefactor * kernel

In [7]:
# ==============================
# Gas maps
# ==============================

def ngasGal(R, z, n0, R0, z0): # cm^-3
    return n0 * np.exp(-np.abs(z)/z0) * np.exp(-(R-rSun)/R0)

def nHI(R, z):
    return ngasGal(R, z, 0.5, 10, 0.2)

def nH2(R, z):
    return ngasGal(R, z, 0.15, 10, 0.05)

def nHII(R, z):
    return ngasGal(R, z, 0.03, 10, 1.0)

def nHe(R, z):
    return ngasGal(R, z, 0.1*(0.5+2*0.15+0.03), 10, 0.2)

In [8]:
# ===============================
# Bremsstrahlung radiating power
# ===============================

def PBrems(Eg, Ee, ngas, *args): # GeV s^-1 GeV^-1
    Eg = np.asarray(Eg)
    Ee = np.asarray(Ee)

    gamma = Ee/me
    epsilon = Eg/Ee

    def dsigmadEg(phi1, phi2):
        prefactor = 3*alpha*sigmaT / (8*np.pi*Eg)
        return prefactor * ((1 + (1-epsilon)**2) * phi1 - 2/3 * (1-epsilon) * phi2)
    
    def phiion(Z):
        return 4 * Z * (Z+1) * (np.log(2*gamma * (1/epsilon-1)) - 1/2)

    phi1HIss, phi2HIss = 45.79, 44.46
    phi1H2ss, phi2H2ss = 2*phi1HIss, 2*phi2HIss
    phi1Hess, phi2Hess = 134.6, 131.4

    if ngas == nHII:
        Pbrems = Eg * ngas(*args) * dsigmadEg(phiion(1), phiion(1))
        Pbrems *= GeVinsec/GeVincm
        return Pbrems
    if ngas == nHI:
        Pss = Eg * ngas(*args) * dsigmadEg(phi1HIss, phi2HIss)
        Psw = Eg * ngas(*args) * dsigmadEg(phiion(1), phiion(1))
    elif ngas == nH2: 
        Pss = Eg * ngas(*args) * dsigmadEg(phi1H2ss, phi2H2ss)
        Psw = Eg * ngas(*args) * dsigmadEg(phiion(2), phiion(2))
    elif ngas == nHe:
        Pss = Eg * ngas(*args) * dsigmadEg(phi1Hess, phi2Hess)
        Psw = Eg * ngas(*args) * dsigmadEg(phiion(2), phiion(2))   

    logg = np.log10(gamma)
    t = (logg - 2.0) / (3.0 - 2.0)
    t = np.clip(t, 0.0, 1.0)
    Pbrems = (1 - t) * Psw + t * Pss

    Pbrems *= GeVinsec/GeVincm

    return Pbrems

### Question 3.1.1 (Section 3.1)
Estimate the IC peak energy
$$E_{\gamma,\mathrm{peak}}^{\mathrm{IC}} = \frac{4\gamma^2 E_{\gamma0}}{1+4\gamma E_{\gamma0}/m_e}, \quad \gamma=E_e/m_e,$$
for $m_{DM}=1$ GeV, 1 TeV, 1 PeV (assuming $E_e\approx m_{DM}$), and target photons:
- CMB: $E_{\gamma0}=0.6$ meV
- IR: $E_{\gamma0}=10$ meV
- Optical: $E_{\gamma0}=1$ eV

In [9]:
# Solution 3.1.1
me_eV = 5.11e5

masses_eV = {
    '1 GeV': 1e9,
    '1 TeV': 1e12,
    '1 PeV': 1e15,
}

targets_eV = {
    'CMB (0.6 meV)': 0.6e-3,
    'IR (10 meV)': 10e-3,
    'Optical (1 eV)': 1.0,
}

def E_ic_peak(Ee_eV, Eg0_eV):
    gamma = Ee_eV / me_eV
    return 4 * gamma**2 * Eg0_eV / (1 + 4 * gamma * Eg0_eV / me_eV)

def band_label(EeV):
    if EeV < 1e2:
        return 'UV/soft X'
    if EeV < 1e5:
        return 'X-ray'
    if EeV < 1e6:
        return 'hard X / soft gamma'
    if EeV < 1e9:
        return 'gamma (MeV-GeV)'
    if EeV < 1e12:
        return 'gamma (GeV-TeV)'
    return 'gamma (TeV-PeV)'

print('Assumption: Ee ~ mDM (e.g. DM DM -> e+e-)')
for mlabel, Ee in masses_eV.items():
    print(f'\n{mlabel}:')
    for tlabel, Eg0 in targets_eV.items():
        Epk = E_ic_peak(Ee, Eg0)
        lam_nm = 1239.841984 / Epk
        print(f'  {tlabel:18s} -> E_peak = {Epk:.4e} eV ({Epk/1e6:.4e} MeV), lambda = {lam_nm:.3e} nm, band = {band_label(Epk)}')

print('\nRelevant observatories by range (from Fig. 1 style):')
print('- X-ray / soft gamma: Chandra, XMM-Newton, NuSTAR, INTEGRAL')
print('- MeV-GeV gamma: COMPTEL (historical), Fermi-LAT, future e-ASTROGAM/AMEGO-like')
print('- GeV-TeV+ gamma: Fermi-LAT, HESS, MAGIC, VERITAS, HAWC, LHAASO, CTA')


Assumption: Ee ~ mDM (e.g. DM DM -> e+e-)

1 GeV:
  CMB (0.6 meV)      -> E_peak = 9.1911e+03 eV (9.1911e-03 MeV), lambda = 1.349e-01 nm, band = X-ray
  IR (10 meV)        -> E_peak = 1.5316e+05 eV (1.5316e-01 MeV), lambda = 8.095e-03 nm, band = hard X / soft gamma
  Optical (1 eV)     -> E_peak = 1.5087e+07 eV (1.5087e+01 MeV), lambda = 8.218e-05 nm, band = gamma (MeV-GeV)

1 TeV:
  CMB (0.6 meV)      -> E_peak = 9.1074e+09 eV (9.1074e+03 MeV), lambda = 1.361e-07 nm, band = gamma (GeV-TeV)
  IR (10 meV)        -> E_peak = 1.3284e+11 eV (1.3284e+05 MeV), lambda = 9.334e-09 nm, band = gamma (GeV-TeV)
  Optical (1 eV)     -> E_peak = 9.3872e+11 eV (9.3872e+05 MeV), lambda = 1.321e-09 nm, band = gamma (GeV-TeV)

1 PeV:
  CMB (0.6 meV)      -> E_peak = 9.0188e+14 eV (9.0188e+08 MeV), lambda = 1.375e-12 nm, band = gamma (TeV-PeV)
  IR (10 meV)        -> E_peak = 9.9351e+14 eV (9.9351e+08 MeV), lambda = 1.248e-12 nm, band = gamma (TeV-PeV)
  Optical (1 eV)     -> E_peak = 9.9993e+14 eV (9.99

### Question 3.1.2 (Section 3.1)
Compute the synchrotron peak energy for $m_{DM}=1$ GeV, 1 TeV, 1 PeV using
$$E^{\rm syn}_{\gamma,\rm peak} = 0.114\,E^{\rm syn}_{\gamma,c},\qquad E^{\rm syn}_{\gamma,c}=\frac{3eB\gamma^2}{2\pi m_e},\quad \gamma=E_e/m_e,$$
with $B=5\,\mu$G and $E_e\approx m_{DM}$ (for DM DM $\to e^+e^-$).

In [12]:
# Solution 3.1.2
import numpy as np

# Constants
me = 5.11e-4            # GeV
alpha = 1/137
e = np.sqrt(4*np.pi*alpha)
GaussinGeV2 = 1.95e-20   # GeV^2 / G
h_eVs = 4.135667696e-15  # eV s
c = 2.99792458e8         # m/s

B_G = 5e-6  # 5 microG

masses_GeV = {'1 GeV': 1.0, '1 TeV': 1e3, '1 PeV': 1e6}

def Esyn_peak_eV(Ee_GeV, B_G):
    gamma = Ee_GeV / me
    E_c_GeV = 3 * e * (B_G * GaussinGeV2) * gamma**2 / (2*np.pi*me)
    E_peak_GeV = 0.114 * E_c_GeV
    return E_peak_GeV * 1e9

def band_label(EeV):
    if EeV < 1e-3:
        return 'radio / microwave'
    if EeV < 1:
        return 'IR / optical'
    if EeV < 1e3:
        return 'UV / soft X-ray'
    if EeV < 1e6:
        return 'X-ray'
    return 'gamma-ray'

print(f'B = {B_G:.2e} G')
for mlabel, mGeV in masses_GeV.items():
    Epk_eV = Esyn_peak_eV(mGeV, B_G)
    nu = Epk_eV / h_eVs
    lam = c / nu
    print(f'{mlabel}: E_peak = {Epk_eV:.4e} eV, nu = {nu:.4e} Hz, lambda = {lam:.4e} m, band = {band_label(Epk_eV)}')

print('\nRelevant observatories by band:')
print('- GeV DM case (radio MHz): radio telescopes (e.g. Green Bank, MeerKAT, SKA)')
print('- TeV DM case (IR/sub-mm): IR/sub-mm facilities (e.g. Planck-like, ALMA-like bands)')
print('- PeV DM case (X-rays): X-ray telescopes (Chandra, XMM-Newton, NuSTAR, Athena-like)')


B = 5.00e-06 G
1 GeV: E_peak = 1.2046e-08 eV, nu = 2.9126e+06 Hz, lambda = 1.0293e+02 m, band = radio / microwave
1 TeV: E_peak = 1.2046e-02 eV, nu = 2.9126e+12 Hz, lambda = 1.0293e-04 m, band = IR / optical
1 PeV: E_peak = 1.2046e+04 eV, nu = 2.9126e+18 Hz, lambda = 1.0293e-10 m, band = X-ray

Relevant observatories by band:
- GeV DM case (radio MHz): radio telescopes (e.g. Green Bank, MeerKAT, SKA)
- TeV DM case (IR/sub-mm): IR/sub-mm facilities (e.g. Planck-like, ALMA-like bands)
- PeV DM case (X-rays): X-ray telescopes (Chandra, XMM-Newton, NuSTAR, Athena-like)
